In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Load the datasets
df1 = pd.read_csv('attached_assets/Maquina_expendedora_1_1764961457195.csv')
df2 = pd.read_csv('attached_assets/Maquina_expendedora_2_1764961457196.csv')

In [ ]:
# --- Data Preprocessing & Cleaning ---

# 1. Standardize columns
df1['datetime'] = pd.to_datetime(df1['datetime'])
df2['datetime'] = pd.to_datetime(df2['datetime'])

# Add machine identifier
df1['machine_id'] = 'Máquina 1'
df2['machine_id'] = 'Máquina 2'

# Combine for unified processing
df_combined = pd.concat([
    df1[['date', 'datetime', 'cash_type', 'money', 'coffee_name', 'machine_id']],
    df2[['date', 'datetime', 'cash_type', 'money', 'coffee_name', 'machine_id']]
])

# 2. Initial Exploration
print("--- Data Info ---")
print(df_combined.info())
print("\n--- Data Head ---")
print(df_combined.head())
print("\n--- Data Shape ---")
print(df_combined.shape)
print("\n--- Data Description ---")
print(df_combined.describe())

# 3. Missing Values Analysis
print("\n--- Missing Values Before Imputation ---")
print(df_combined.isnull().sum())

# Handle missing values if any (using SimpleImputer for demonstration)
# For numerical columns: replace with mean
num_imputer = SimpleImputer(strategy='mean')
if df_combined['money'].isnull().any():
    df_combined['money'] = num_imputer.fit_transform(df_combined[['money']])

# For categorical columns: replace with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
if df_combined['cash_type'].isnull().any():
    df_combined['cash_type'] = cat_imputer.fit_transform(df_combined[['cash_type']]).ravel()

# 4. Duplicate Values
print("\n--- Duplicate Values ---")
duplicates = df_combined.duplicated().sum()
print(f"Found {duplicates} duplicates")
if duplicates > 0:
    df_combined = df_combined.drop_duplicates()
    print("Duplicates dropped.")

# 5. Outlier Analysis (IQR Method)
print("\n--- Outlier Analysis (Money) ---")
Q1 = df_combined['money'].quantile(0.25)
Q3 = df_combined['money'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_combined[(df_combined['money'] < lower_bound) | (df_combined['money'] > upper_bound)]
print(f"Number of outliers detected in 'money': {len(outliers)}")
print(f"IQR: {IQR}, Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")

# 6. Statistical Metrics
print("\n--- Statistical Metrics (Money) ---")
stats = {
    'min': df_combined['money'].min(),
    'q1': Q1,
    'median': df_combined['money'].median(),
    'q3': Q3,
    'max': df_combined['money'].max(),
    'std_dev': df_combined['money'].std()
}
print(stats)

# 7. Normalization (Demonstration)
# StandardScaler (Z-score normalization)
scaler = StandardScaler()
df_combined['money_standardized'] = scaler.fit_transform(df_combined[['money']])

# MinMaxScaler
minmax = MinMaxScaler()
df_combined['money_normalized'] = minmax.fit_transform(df_combined[['money']])

print("\n--- Normalized Data Sample ---")
print(df_combined[['money', 'money_standardized', 'money_normalized']].head())

In [ ]:
# --- Analysis & Visualization ---

# 1. Total Revenue Comparison
total_revenue = df_combined.groupby('machine_id')['money'].sum().reset_index()
print("\n--- Ingresos Totales por Máquina ---")
print(total_revenue)

# 2. Sales by Hour of Day
df_combined['hour'] = df_combined['datetime'].dt.hour
hourly_sales = df_combined.groupby(['machine_id', 'hour'])['money'].sum().reset_index()

# 3. Top Products
top_products = df_combined.groupby(['machine_id', 'coffee_name'])['money'].sum().reset_index()
top_products = top_products.sort_values(['machine_id', 'money'], ascending=[True, False])
top_products = top_products.groupby('machine_id').head(5)

# Generate Plots
sns.set_theme(style="whitegrid")

# Plot 1: Revenue Comparison
plt.figure(figsize=(8, 6))
sns.barplot(data=total_revenue, x='machine_id', y='money', palette='viridis')
plt.title('Ingresos Totales: Máquina 1 vs Máquina 2')
plt.ylabel('Ingresos ($)')
plt.savefig('client/public/analysis_revenue.png')

# Plot 2: Hourly Sales Trends
plt.figure(figsize=(12, 6))
sns.lineplot(data=hourly_sales, x='hour', y='money', hue='machine_id', marker='o')
plt.title('Tendencia de Ventas por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Ventas ($)')
plt.xticks(range(0, 24))
plt.grid(True)
plt.savefig('client/public/analysis_hourly.png')

# Plot 3: Top Products Comparison
plt.figure(figsize=(12, 8))
sns.barplot(data=top_products, y='coffee_name', x='money', hue='machine_id')
plt.title('Top 5 Productos por Ingresos Generados')
plt.xlabel('Ingresos ($)')
plt.ylabel('Producto')
plt.tight_layout()
plt.savefig('client/public/analysis_products.png')

print("\nCoffee Analysis complete. Images saved to client/public/")